### Sentiment Analysis on IMDb Dataset using SVM

Sentiment analysis is the process of determining the emotional tone behind a body of text. We'll use the IMDb movie review dataset, which contains movie reviews labeled as positive or negative, to train an SVM model to classify the sentiment of new reviews.

#### Step 1: Load the IMDb Dataset

First, we need to load the IMDb movie review dataset. We can conveniently do this using `tensorflow_datasets`.

In [1]:
import tensorflow_datasets as tfds
import numpy as np

# Load the IMDb movie reviews dataset
# The dataset comes pre-split into training and testing sets.
(train_data, test_data), info = tfds.load(
    name="imdb_reviews",
    split=(tfds.Split.TRAIN, tfds.Split.TEST),
    with_info=True,
    as_supervised=True # Returns (text, label) tuples
)

print(f"Number of training samples: {info.splits['train'].num_examples}")
print(f"Number of test samples: {info.splits['test'].num_examples}")

# Convert the dataset to lists of text and labels for easier processing with scikit-learn
def convert_to_list(tf_dataset):
    texts = []
    labels = []
    for text_tensor, label_tensor in tf_dataset:
        texts.append(text_tensor.numpy().decode('utf-8'))
        labels.append(label_tensor.numpy())
    return texts, labels

train_texts, train_labels = convert_to_list(train_data)
test_texts, test_labels = convert_to_list(test_data)

# Display the first few examples to understand the data format
print("\nFirst 5 training examples:")
for i in range(5):
    print(f"Review: {train_texts[i][:100]}...") # Show first 100 chars
    print(f"Label: {'Positive' if train_labels[i] == 1 else 'Negative'}")
    print("---------------------------------")


Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Generating splits...:   0%|          | 0/3 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/imdb_reviews/plain_text/incomplete.9B4X7U_1.0.0/imdb_reviews-train.tfrecor…

Generating test examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/imdb_reviews/plain_text/incomplete.9B4X7U_1.0.0/imdb_reviews-test.tfrecord…

Generating unsupervised examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/imdb_reviews/plain_text/incomplete.9B4X7U_1.0.0/imdb_reviews-unsupervised.…

Dataset imdb_reviews downloaded and prepared to /root/tensorflow_datasets/imdb_reviews/plain_text/1.0.0. Subsequent calls will reuse this data.
Number of training samples: 25000
Number of test samples: 25000

First 5 training examples:
Review: This was an absolutely terrible movie. Don't be lured in by Christopher Walken or Michael Ironside. ...
Label: Negative
---------------------------------
Review: I have been known to fall asleep during films, but this is usually due to a combination of things in...
Label: Negative
---------------------------------
Review: Mann photographs the Alberta Rocky Mountains in a superb fashion, and Jimmy Stewart and Walter Brenn...
Label: Negative
---------------------------------
Review: This is the kind of film for a snowy Sunday afternoon when the rest of the world can go ahead with i...
Label: Positive
---------------------------------
Review: As others have mentioned, all the women that go nude in this film are mostly absolutely gorgeous. Th...
Labe

#### Step 2: Text Preprocessing and Feature Extraction

Before we can train an SVM model, we need to convert the text reviews into numerical features. This typically involves:
1.  **Tokenization**: Breaking text into individual words or subword units.
2.  **Removing Stop Words**: Eliminating common words (e.g., 'the', 'is', 'a') that don't carry much sentiment.
3.  **Stemming/Lemmatization**: Reducing words to their root form (e.g., 'running' -> 'run').
4.  **Vectorization**: Converting the processed text into numerical vectors. A common technique for this is TF-IDF (Term Frequency-Inverse Document Frequency).

We will use `TfidfVectorizer` from `scikit-learn` to perform both tokenization (with some basic preprocessing) and TF-IDF weighting, converting our text data into a matrix of TF-IDF features.

In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Initialize TF-IDF Vectorizer
# max_features: consider only the top N words by frequency
# min_df: ignore words that appear in less than X documents
# max_df: ignore words that appear in more than X% of documents
vectorizer = TfidfVectorizer(max_features=5000, min_df=5, max_df=0.8, stop_words='english')

# Fit the vectorizer on the training data and transform both training and test data
X_train = vectorizer.fit_transform(train_texts)
X_test = vectorizer.transform(test_texts)

# Display the shape of the resulting feature matrices
print(f"Shape of X_train (TF-IDF features for training): {X_train.shape}")
print(f"Shape of X_test (TF-IDF features for testing): {X_test.shape}")

Shape of X_train (TF-IDF features for training): (25000, 5000)
Shape of X_test (TF-IDF features for testing): (25000, 5000)


#### Step 3: Train the SVM Model

With our text data converted into numerical features, we can now train a Support Vector Machine (SVM) classifier. SVMs are powerful and effective for classification tasks, especially in text classification. We'll use `LinearSVC` from `scikit-learn`, which is suitable for large datasets and performs well with TF-IDF features.

In [3]:
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, accuracy_score

# Initialize the Linear SVM model
svm_model = LinearSVC(random_state=42)

# Train the SVM model using the training data
print("Training the SVM model...")
svm_model.fit(X_train, train_labels)
print("SVM model training complete.")

Training the SVM model...
SVM model training complete.


#### Step 4: Evaluate the Model

After training, it's crucial to evaluate the model's performance on the test set to see how well it generalizes to new, unseen data. We'll use `classification_report` and `accuracy_score` from `scikit-learn` to assess our SVM model.

In [4]:
# Make predictions on the test set
y_pred = svm_model.predict(X_test)

# Evaluate the model
print("Classification Report:")
print(classification_report(test_labels, y_pred))

print(f"\nAccuracy Score: {accuracy_score(test_labels, y_pred):.4f}")

Classification Report:
              precision    recall  f1-score   support

           0       0.86      0.87      0.86     12500
           1       0.87      0.86      0.86     12500

    accuracy                           0.86     25000
   macro avg       0.86      0.86      0.86     25000
weighted avg       0.86      0.86      0.86     25000


Accuracy Score: 0.8613
